In [ ]:
%load_ext autoreload
%autoreload 2

# Reach grid from forward kinematics

Same reach set as `reach-grid-limits.ipynb`, built without inverse kinematics: grid the
joint space (within the limits), run forward kinematics, and collect the feet in the voxel
grid. Two versions:

- **point grid**: each forwarded joint-grid vertex marks the voxel it lands in (nearest
  vertex). Leaves holes when the joint grid is coarse.
- **hexahedron grid**: each joint-grid *cell* is forwarded at its 8 corners; the deformed
  box they span (split into 6 tetrahedra, consistently across neighbours) marks every voxel
  whose centre lies inside it. No holes at any joint resolution; error only from flat faces
  instead of curved ones (~ r dtheta^2 / 8).

Both keep a representative theta per voxel (the vertex's, or the cell centre's). Compared
against the exact IK grid (`reach_sdf.reach_mask`), voxel by voxel. The body keep-out box is
applied to voxel centres, as in the IK grid.

## Findings (2026-10-09)

Setup: climb leg (0.05 / 0.15 / 0.20 m), limits coxa +-90, femur +-100, knee +-160 deg, body
keep-out box; laptop CPU. "Missed" / "extra": voxels vs. the exact IK grid.

| voxel | joint spacing | joint vertices | FK | point grid | hexahedron grid | point missed / extra | hex missed / extra |
|---|---|---|---|---|---|---|---|
| 1 cm | 4 deg | 0.2M | 0.1 s | 0.01 s | 0.6 s | 38% / 3.2% | 0.22% / 0 |
| 1 cm | 2 deg | 1.5M | 0.3 s | 0.06 s | 1.6 s | 0.9% / 5.5% | 0.07% / 0 |
| 1 cm | 1 deg | 11.8M | 1.2 s | 0.5 s | 7.4 s | 0 / 6.7% | 0.03% / 0 |
| 1 cm | 0.5 deg | 93M | 9.6 s | 4.0 s | 62 s | 0 / 7.2% | 0.03% / 0 |
| 5 mm | 4 deg | 0.2M | 0.02 s | 0.01 s | 2.4 s | 87% / 0.6% | 0.19% / 0.01% |
| 5 mm | 2 deg | 1.5M | 0.15 s | 0.06 s | 3.3 s | 39% / 1.6% | 0.04% / 0 |
| 5 mm | 1 deg | 11.8M | 1.2 s | 0.5 s | 11.5 s | 0.9% / 2.7% | 0.01% / 0 |
| 5 mm | 0.5 deg | 93M | 9.3 s | 3.9 s | 61 s | 0 / 3.3% | 0.01% / 0 |

(IK grid at 5 mm: 0.9 s.)

- **Hexahedron grid**: no holes at any joint spacing, essentially no extra voxels; the few
  missed ones are boundary voxels (flat faces instead of curved, ~ r dtheta^2 / 8, 0.25 mm at
  4 deg). Cost scales with the number of joint cells, so a coarse joint grid (2-4 deg) is the
  sweet spot: 5 mm voxels at 4 deg in 2.4 s, 0.19% missed.
- **Point grid**: holes unless dtheta <~ h / r (at 5 mm none only from 0.5 deg), and always an
  outward bloat of 3-7% (nearest voxel vertex). Fast once fine enough.
- **Representative theta**: its foot lands within ~2 mm (median, max 5.6 mm) of the voxel
  centre (5 mm, 1 deg).
- **Signed distance** from the hexahedron grid vs. the IK one: median 0 mm, max 5 mm.
- **Tie on limit planes**: voxel centres lying exactly on a face of the reach set (the plane
  x = 0 at the coxa limit 90 deg) were missed with a tight barycentric tolerance (1e-9),
  since float32 FK puts the face ~1e-8 m off; eps = 1e-4 fixes it.
- **Memory**: `cell_corners` materializes all 8 corners of every cell; the 0.5 deg rows peaked
  at ~33 GB. Chunk along the first joint axis before going finer (or to 4D).

### Idea: 4-DOF legs, zonotopes (not tried yet)

A 4D joint cell can't be split into tetrahedra in 3D; its image is a projection. Over a small
cell FK is nearly affine, $FK(\theta_c + \delta) \approx FK(\theta_c) + J\delta$ ($J$: 3 x 4
Jacobian), so the cell's image is approximately a **zonotope**: centre $c = FK(\theta_c)$,
generators $g_i = J_{:,i}\,\Delta\theta_i$ (or the averaged forwarded cell edges). It is
convex and equals the convex hull of the 16 forwarded corners; curvature adds ~ $r\,\Delta\theta^2/8$.

No hull algorithm needed: the facet normals are $n_{ij} = g_i \times g_j$ (6 pairs, <= 12
facets), and $p$ is inside iff for every pair $i < j$

$$
\lvert n_{ij}^\top (p - c) \rvert \le \tfrac{1}{2} \sum_k \lvert n_{ij}^\top g_k \rvert
$$

6 fixed-size inequalities per candidate voxel centre, JAX-friendly.

- Not watertight: neighbours come from different linearizations, so they overlap (harmless)
  or leave hairline gaps ~ the curvature error; pad the right-hand side by that bound.
- Degenerate cells (near singularities): skip pairs with $n_{ij} \approx 0$; a flat zonotope
  marks nothing, its neighbours cover the region.
- Cost estimate: at 4 deg ~1e7 cells x K^3 candidates x 6 inequalities, minutes on CPU.
  (Extrapolated from the 3D rates: FK ~0.1 us / vertex, hexahedron marking ~0.65 us / cell at
  K = 1; 1e10 cells, i.e. 0.5 deg in 4D, would be ~2 h on CPU.)
- Alternative: slice the redundant joint at n4 values, run the 3D hexahedron grid per slice,
  take the union (boundary blurred by ~ r dtheta4). Same order of cost.
- In 3D the same test works (3 generators: a parallelepiped), but the tetrahedron split tiles
  exactly, so it stays the choice there.


In [ ]:
import time

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from controlkit.kinematics import Leg3DOF, Robot, radial_mounts
import controlkit.rerun_viz as rrv

from lab.gait_graph.planner.climb_model.config import MjModelCfg
from lab.kinematics.reach_sdf import Keepout, ReachSDF, reach_mask, signed_distance_grid

In [ ]:
# Shoulder at the origin. cfg only seeds the parameters; override them freely.
cfg = MjModelCfg()

LENGTHS = jnp.asarray(cfg.leg_lengths)                               # coxa, femur, tibia (m)
LIMITS = jnp.deg2rad(jnp.array([[-90, 90], [-100, 100], [-160, 160]]))   # (3, 2) rad
leg = Leg3DOF.from_lengths(LENGTHS, LIMITS)
reach = float(LENGTHS.sum())                                         # grid half-width (m)

FILTER_BODY = True   # drop voxels whose centre lies inside the body box

# Body as a keep-out box, placed as if this were the right front leg (mount at 315 deg).
mounts = radial_mounts(cfg.num_legs, radius=cfg.mount_radius)
body_center, body_half = Robot(mounts=mounts, leg=leg).mount_box(half_height=cfg.body_half_height)
keepout = Keepout(mounts[3].inverse(), body_center, body_half)

print("lengths:", LENGTHS, " reach:", reach)
print("limits (deg):", np.rad2deg(np.asarray(LIMITS)).round().tolist())

## Grids

In [ ]:
def voxel_axis(h):
    # Voxel-grid vertex coordinates along each axis, as reach_sdf.reach_mask.
    n = int(round(2 * reach / h)) + 1
    return np.linspace(-reach, reach, n)

def joint_axes(dtheta):
    # Joint-grid vertices per joint, endpoints at the limits, spacing <= dtheta (rad).
    lim = np.asarray(LIMITS)
    return [np.linspace(lo, hi, int(np.ceil((hi - lo) / dtheta)) + 1) for lo, hi in lim]

def body_free(axis):
    # (n, n, n) bool: voxel centre outside the body box (all True without the filter).
    n = len(axis)
    if not FILTER_BODY:
        return np.ones((n, n, n), bool)
    X = np.stack(np.meshgrid(axis, axis, axis, indexing="ij"), -1).reshape(-1, 3)
    inside = jax.jit(jax.vmap(keepout.contains))(jnp.asarray(X, jnp.float32))
    return ~np.asarray(inside).reshape(n, n, n)

_foot = jax.jit(jax.vmap(lambda th: leg.forward(th).translation()[-1]))

def forward_grid(axes, chunk=1 << 20):
    # Feet at every joint-grid vertex: (n1, n2, n3, 3), and the angles (n1, n2, n3, 3).
    TH = np.stack(np.meshgrid(*axes, indexing="ij"), -1).astype(np.float32)
    flat = TH.reshape(-1, 3)
    F = np.concatenate([np.asarray(_foot(jnp.asarray(flat[k:k + chunk])))
                        for k in range(0, len(flat), chunk)])
    return F.reshape(TH.shape), TH

### Point grid

In [ ]:
def point_grid(F, TH, axis):
    # Each forwarded vertex marks its nearest voxel vertex. Returns mask, theta_rep, hits.
    n, h = len(axis), axis[1] - axis[0]
    idx = np.rint((F.reshape(-1, 3) - axis[0]) / h).astype(np.int64)
    ok = np.all((idx >= 0) & (idx < n), -1)
    flat = np.ravel_multi_index(idx[ok].T, (n, n, n))
    hits = np.bincount(flat, minlength=n ** 3).reshape(n, n, n)
    theta = np.full((n ** 3, 3), np.nan, np.float32)
    theta[flat] = TH.reshape(-1, 3)[ok]                     # last write wins: a representative
    return hits > 0, theta.reshape(n, n, n, 3), hits

### Hexahedron grid

Cell corners ordered by bits `(a, b, c)` -> index `4a + 2b + c` (offsets along joints 1, 2,
3). Split along the diagonal 0-7 into 6 tetrahedra (Kuhn / Freudenthal split): neighbouring
cells cut their shared face along the same diagonal, so the tetrahedra tile the image without
gaps. Candidates per cell: the voxel centres in the hexahedron's bounding box, a fixed
`K x K x K` block (K = the largest extent over all cells), masked to the box.

In [ ]:
TETS = np.array([[0, 4, 6, 7], [0, 6, 2, 7], [0, 2, 3, 7], [0, 3, 1, 7], [0, 1, 5, 7], [0, 5, 4, 7]])

def cell_corners(A):
    # (n1, n2, n3, ...) vertex array -> (n1-1, n2-1, n3-1, 8, ...) cell corners.
    s = [A[a:A.shape[0] - 1 + a, b:A.shape[1] - 1 + b, c:A.shape[2] - 1 + c]
         for a in (0, 1) for b in (0, 1) for c in (0, 1)]
    return np.stack(s, 3)

def _in_tets(P, V, eps=1e-4):
    # P (Q, 3) candidate points, V (8, 3) hexahedron corners -> (Q,) inside any tetrahedron.
    # eps (barycentric, ~ eps x cell size in metres) catches voxel centres lying exactly on a
    # face that float32 FK puts a hair off (e.g. the plane x = 0 at a coxa limit of 90 deg).
    T = V[TETS]                                            # (6, 4, 3)
    M = jnp.transpose(T[:, 1:] - T[:, :1], (0, 2, 1))      # (6, 3, 3), columns v1-v0, v2-v0, v3-v0
    det = jnp.linalg.det(M)
    good = jnp.abs(det) > 1e-18                            # skip flat tetrahedra
    Minv = jnp.linalg.inv(jnp.where(good[:, None, None], M, jnp.eye(3)))
    lam = jnp.einsum("tij,tqj->tqi", Minv, P[None] - T[:, :1])   # (6, Q, 3)
    inside = (lam >= -eps).all(-1) & (lam.sum(-1) <= 1 + eps) & good[:, None]
    return inside.any(0)

def _cell_marks(V, lo, h, n, K):
    # One cell: (K^3, 3) candidate voxel indices and whether each is marked.
    imin = jnp.ceil((V.min(0) - lo) / h).astype(jnp.int32)
    imax = jnp.floor((V.max(0) - lo) / h).astype(jnp.int32)
    off = jnp.stack(jnp.meshgrid(*[jnp.arange(K)] * 3, indexing="ij"), -1).reshape(-1, 3)
    idx = imin + off
    valid = (idx <= imax).all(-1) & (idx >= 0).all(-1) & (idx < n).all(-1)
    return idx, valid & _in_tets(lo + idx * h, V)

def hex_grid(F, TH, axis, max_candidates=1 << 23):
    # Mark voxel centres inside the forwarded joint cells. Returns mask, theta_rep, hits, K.
    n, h, lo = len(axis), float(axis[1] - axis[0]), float(axis[0])
    CF = cell_corners(F).reshape(-1, 8, 3)
    CT = cell_corners(TH).reshape(-1, 8, 3).mean(1)        # cell-centre angles
    ext = np.floor((CF.max(1) - lo) / h) - np.ceil((CF.min(1) - lo) / h) + 1
    K = max(int(ext.max()), 1)
    marks = jax.jit(jax.vmap(lambda V: _cell_marks(V, lo, h, n, K)))
    chunk = max(max_candidates // K ** 3, 1)
    hits = np.zeros(n ** 3, np.int64)
    theta = np.full((n ** 3, 3), np.nan, np.float32)
    for k in range(0, len(CF), chunk):
        idx, m = marks(jnp.asarray(CF[k:k + chunk]))
        idx, m = np.asarray(idx), np.asarray(m)
        cell = np.broadcast_to(np.arange(idx.shape[0])[:, None], m.shape)[m]
        flat = np.ravel_multi_index(idx[m].T, (n, n, n))
        np.add.at(hits, flat, 1)
        theta[flat] = CT[k + cell]
    return hits.reshape(n, n, n) > 0, theta.reshape(n, n, n, 3), hits.reshape(n, n, n), K

## One build

In [ ]:
H = 0.005                 # voxel size (m)
DTHETA = np.deg2rad(1.0)  # joint-grid spacing (rad)

axis = voxel_axis(H)
n = len(axis)
free = body_free(axis)
axes = joint_axes(DTHETA)
print(f"voxels {n}^3 = {n ** 3:,} | joint grid {' x '.join(str(len(a)) for a in axes)} "
      f"= {np.prod([len(a) for a in axes]):,} vertices")

t0 = time.perf_counter(); F, TH = forward_grid(axes); t_fk = time.perf_counter() - t0
t0 = time.perf_counter(); pmask, ptheta, phits = point_grid(F, TH, axis); t_pt = time.perf_counter() - t0
t0 = time.perf_counter(); hmask, htheta, hhits, K = hex_grid(F, TH, axis); t_hex = time.perf_counter() - t0
pmask &= free; hmask &= free
print(f"forward kinematics {t_fk:.2f}s | point grid {t_pt:.2f}s | hexahedron grid {t_hex:.2f}s (K = {K})")

t0 = time.perf_counter()
imask, _ = reach_mask(leg, H, keepout=keepout if FILTER_BODY else None)   # exact, from IK
print(f"IK grid {time.perf_counter() - t0:.2f}s")

In [ ]:
def compare(mask, ref):
    # Voxel counts: (reachable in ref, missed (holes), extra).
    return int(ref.sum()), int((ref & ~mask).sum()), int((mask & ~ref).sum())

for name, mask in (("point", pmask), ("hexahedron", hmask)):
    r, miss, extra = compare(mask, imask)
    print(f"{name:10s}: {int(mask.sum()):,} voxels | vs IK ({r:,}): missed {miss:,} ({miss / r:.2%}), "
          f"extra {extra:,} ({extra / r:.2%})")

# Representative angles: their foot should land near the voxel centre.
X = np.stack(np.meshgrid(axis, axis, axis, indexing="ij"), -1)
for name, mask, th in (("point", pmask, ptheta), ("hexahedron", hmask, htheta)):
    feet = np.asarray(_foot(jnp.asarray(th[mask])))
    err = np.linalg.norm(feet - X[mask], axis=-1)
    print(f"{name:10s}: |foot(theta_rep) - voxel centre| median {np.median(err) * 1e3:.2f} mm, "
          f"max {err.max() * 1e3:.2f} mm")

In [ ]:
# Where the masks disagree with IK, on the z = 0 slice.
iz = int(np.argmin(np.abs(axis)))
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, (name, mask) in zip(ax, (("IK", imask), ("point", pmask), ("hexahedron", hmask))):
    img = np.zeros((n, n, 3))
    img[imask[:, :, iz] & mask[:, :, iz]] = (0.6, 0.6, 0.6)   # both
    img[imask[:, :, iz] & ~mask[:, :, iz]] = (0.9, 0.2, 0.2)  # missed
    img[~imask[:, :, iz] & mask[:, :, iz]] = (0.2, 0.4, 0.9)  # extra
    a.imshow(np.transpose(img, (1, 0, 2)), origin="lower", extent=[axis[0], axis[-1]] * 2)
    a.set_title(f"{name}: grey both, red missed, blue extra"); a.set_xlabel("x"); a.set_ylabel("y")

In [ ]:
rrv.init_viewer(connect=8812)
rrv.set_time(0, timeline="t")
for name, mask in (("point", pmask), ("hexahedron", hmask)):
    rrv.log_points(f"{name}/missed", X[imask & ~mask], radius=0.002, color=(230, 50, 50))
    rrv.log_points(f"{name}/extra", X[mask & ~imask], radius=0.002, color=(50, 100, 230))
rrv.log_points("ik/reachable", X[imask], radius=0.001, color=(150, 150, 150))

## Signed distance from the hexahedron grid

In [ ]:
PAD = 4
sdf_fk = ReachSDF(D=jnp.asarray(signed_distance_grid(np.pad(hmask, PAD), H)), lo=float(axis[0]) - PAD * H, h=H)
sdf_ik = ReachSDF(D=jnp.asarray(signed_distance_grid(np.pad(imask, PAD), H)), lo=float(axis[0]) - PAD * H, h=H)
P = jax.random.uniform(jax.random.PRNGKey(1), (200_000, 3), minval=-reach, maxval=reach)
d_fk, d_ik = jax.vmap(sdf_fk.sdf)(P), jax.vmap(sdf_ik.sdf)(P)
print(f"|sdf_fk - sdf_ik|: median {float(jnp.median(jnp.abs(d_fk - d_ik))) * 1e3:.2f} mm, "
      f"max {float(jnp.abs(d_fk - d_ik).max()) * 1e3:.2f} mm")

## Timing sweep

Joint spacing x voxel size. Time per stage, and the voxel errors against the IK grid.

In [ ]:
DTHETAS_DEG = (4.0, 2.0, 1.0, 0.5)
HS = (0.01, 0.005)

rows = []
for H_ in HS:
    ax_ = voxel_axis(H_)
    free_ = body_free(ax_)
    ref, _ = reach_mask(leg, H_, keepout=keepout if FILTER_BODY else None)
    for dd in DTHETAS_DEG:
        axes_ = joint_axes(np.deg2rad(dd))
        t0 = time.perf_counter(); F_, TH_ = forward_grid(axes_); t_fk = time.perf_counter() - t0
        t0 = time.perf_counter(); pm, *_ = point_grid(F_, TH_, ax_); t_pt = time.perf_counter() - t0
        t0 = time.perf_counter(); hm, _, _, K_ = hex_grid(F_, TH_, ax_); t_hx = time.perf_counter() - t0
        pm &= free_; hm &= free_
        r, pmiss, pextra = compare(pm, ref)
        _, hmiss, hextra = compare(hm, ref)
        rows.append(dict(h_mm=H_ * 1e3, dtheta_deg=dd, joint_vertices=int(np.prod([len(a) for a in axes_])),
                         t_fk=t_fk, t_point=t_pt, t_hex=t_hx, K=K_, point_missed=pmiss / r,
                         point_extra=pextra / r, hex_missed=hmiss / r, hex_extra=hextra / r))
        print(f"h {H_ * 1e3:4g} mm  dtheta {dd:3g} deg | joint vertices {rows[-1]['joint_vertices']:>11,} | "
              f"FK {t_fk:6.2f}s  point {t_pt:6.2f}s  hex {t_hx:6.2f}s (K {K_}) | "
              f"point missed {pmiss / r:6.2%} extra {pextra / r:6.2%} | hex missed {hmiss / r:6.2%} extra {hextra / r:6.2%}")